In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("NYC_TLC_DataWarehouse")
    .enableHiveSupport()
    .config("spark.sql.warehouse.dir", "/user/hive/warehouse")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.memory", "1500m")
    .config("spark.executor.memory", "1500m")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.default.parallelism", "4")
    .config("spark.memory.fraction", "0.6")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(spark.version)
print(spark.conf.get("spark.sql.catalogImplementation"))

2026-09-03 09:27:18,229 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
2026-09-03 09:27:19,328 WARN util.Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


3.1.2
hive


In [2]:
BASE_PATH = "/user/student/cleanedUber"

df_trips_raw = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(f"{BASE_PATH}/trips")
)
df_trips_raw.printSchema()
df_trips_raw.show(3, truncate=False)
print(df_trips_raw.count())

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp (nullable = true)
 |-- on_scene_datetime: timestamp (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_flag: strin

2026-09-03 09:27:25,105 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+-----+----------+-------------------+-----------------+------------------+----------------+--------------+------------------+-------+--------------------+---------------------+-------------------+--------------------+----------------------+----------------------+---------------------+----------------------+--------------------+---------------------+-----------------------+-----------------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|request_datetime   |on_scene_datetime  |pickup_datetime    |dropoff_datetime   |PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls|bcf |sales_tax|congestion_surcharge|airport_fee|tips |driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wa

In [3]:
def smart_read(path, name):
    attempts = [
        ("parquet", {}),
        ("json", {}),
        ("csv", {"header": "true", "inferSchema": "true"}),
    ]
    last_error = None
    for fmt, options in attempts:
        try:
            df = spark.read.format(fmt).options(**options).load(path)
            df.limit(1).collect()
            print(name, fmt, path)
            return df
        except Exception as e:
            last_error = e
            continue
    raise RuntimeError(f"{name} {path} {last_error}")

df_weather_raw = smart_read(f"{BASE_PATH}/weather", "weather")
df_lookup_raw  = smart_read(f"{BASE_PATH}/lookup", "lookup")

df_weather_raw.printSchema()
df_weather_raw.show(3, truncate=False)
print(df_weather_raw.count())

df_lookup_raw.printSchema()
df_lookup_raw.show(3, truncate=False)
print(df_lookup_raw.count())

2026-09-03 09:27:28,080 ERROR executor.Executor: Exception in task 0.0 in stage 4.0 (TID 20)
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:301)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:375)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:450)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:496)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:490)
	at org.apache.spark.sql.execution.datasources.SchemaMergeUtils$.$anonfun$mergeSchemasInParallel$2(SchemaMergeUtils.scala:75)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2(RDD.scala:863)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2$adapted(RDD.scala:863)
	at org.apa

weather csv /user/student/cleanedUber/weather


2026-09-03 09:27:29,369 ERROR executor.Executor: Exception in task 0.0 in stage 9.0 (TID 25)
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:301)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:375)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:450)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:496)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:490)
	at org.apache.spark.sql.execution.datasources.SchemaMergeUtils$.$anonfun$mergeSchemasInParallel$2(SchemaMergeUtils.scala:75)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2(RDD.scala:863)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2$adapted(RDD.scala:863)
	at org.apa

lookup csv /user/student/cleanedUber/lookup
root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- Date: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)

+---------+---------+----------------+-------------------+--------------+-------------+--------+----------+----+----+-----+---+
|latitude |longitude|timezone        |time               |temperature_2m|precipitation|snowfall|Date      |hour|year|month|day|
+---------+---------+----------------+-------------------+--------------+-------------+--------+----------+----+----+-----+---+
|40.738136|-74.04254|America/New_York|2025-11-30 15:00:00|4.0           |0.0          |0.0 

In [4]:
TRIPS_COLS = {
    "hvfhs_license_num":     "hvfhs_license_num",
    "dispatching_base_num":  "dispatching_base_num",
    "originating_base_num":  "originating_base_num",
    "request_datetime":      "request_datetime",
    "on_scene_datetime":     "on_scene_datetime",
    "pickup_datetime":       "pickup_datetime",
    "dropoff_datetime":      "dropoff_datetime",
    "pickup_location_id":    "PULocationID",
    "dropoff_location_id":   "DOLocationID",
    "trip_miles":            "trip_miles",
    "trip_time":             "trip_time",
    "base_passenger_fare":   "base_passenger_fare",
    "tolls":                 "tolls",
    "bcf":                   "bcf",
    "sales_tax":             "sales_tax",
    "congestion_surcharge":  "congestion_surcharge",
    "airport_fee":           "airport_fee",
    "tips":                  "tips",
    "driver_pay":            "driver_pay",
    "cbd_congestion_fee":    "cbd_congestion_fee",
    "shared_request_flag":   "shared_request_flag",
    "shared_match_flag":     "shared_match_flag",
    "access_a_ride_flag":    "access_a_ride_flag",
    "wav_request_flag":      "wav_request_flag",
    "wav_match_flag":        "wav_match_flag",
}

WEATHER_COLS = {
    "date":            "Date",
    "hour":            "hour",
    "temperature_2m":  "temperature_2m",
    "precipitation":   "precipitation",
    "snowfall":        "snowfall",
    "timezone":        "timezone",
}

LOOKUP_COLS = {
    "location_id":  "LocationID",
    "borough":      "Borough",
    "zone":         "Zone",
    "service_zone": "service_zone",
}

def check_columns(df, mapping, df_name):
    missing = [src for src in mapping.values() if src not in df.columns]
    if missing:
        print(df_name, "MISSING", missing)
        print(df_name, "ACTUAL", df.columns)
    else:
        print(df_name, "OK")

check_columns(df_trips_raw, TRIPS_COLS, "trips")
check_columns(df_weather_raw, WEATHER_COLS, "weather")
check_columns(df_lookup_raw, LOOKUP_COLS, "lookup")

trips OK
weather OK
lookup OK


In [5]:
spark.sql("CREATE DATABASE IF NOT EXISTS nyc_tlc_dwh")
spark.sql("USE nyc_tlc_dwh")
print(spark.catalog.currentDatabase())

2026-09-03 09:27:31,674 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-03 09:27:31,676 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist


nyc_tlc_dwh


2026-09-03 09:27:33,419 WARN metastore.ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
2026-09-03 09:27:33,429 ERROR metastore.RetryingHMSHandler: AlreadyExistsException(message:Database nyc_tlc_dwh already exists)
	at org.apache.hadoop.hive.metastore.HiveMetaStore$HMSHandler.create_database(HiveMetaStore.java:925)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at sun.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.lang.reflect.Method.invoke(Method.java:498)
	at org.apache.hadoop.hive.metastore.RetryingHMSHandler.invokeInternal(RetryingHMSHandler.java:148)
	at org.apache.hadoop.hive.metastore.RetryingHMSHandler.invoke(RetryingHMSHandler.java:107)
	at com.sun.proxy.$Proxy35.create_database(Unknown Source)
	at org.apache.hadoop.hive.metastore.HiveMetaStoreClient.createDatabase(HiveMetaStoreClient.java:727)
	at sun.

In [6]:
df_trips = df_trips_raw.select(
    *[F.col(TRIPS_COLS[c]).alias(c) for c in TRIPS_COLS]
)

pickup_dates = (
    df_trips
    .select(F.to_date(F.col("pickup_datetime")).alias("full_date"))
    .where(F.col("full_date").isNotNull())
    .distinct()
)

dim_date = (
    pickup_dates
    .withColumn("date_id", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("year", F.year("full_date"))
    .withColumn("is_weekend", F.col("day_of_week").isin(1, 7))
    .select("date_id", "full_date", "day_name", "day_of_week", "month",
            "month_name", "quarter", "year", "is_weekend")
)

dim_date.show(5)
print(dim_date.count())

+--------+----------+---------+-----------+-----+----------+-------+----+----------+
| date_id| full_date| day_name|day_of_week|month|month_name|quarter|year|is_weekend|
+--------+----------+---------+-----------+-----+----------+-------+----+----------+
|20251201|2025-12-01|   Monday|          2|   12|  December|      4|2025|     false|
|20251212|2025-12-12|   Friday|          6|   12|  December|      4|2025|     false|
|20251221|2025-12-21|   Sunday|          1|   12|  December|      4|2025|      true|
|20251223|2025-12-23|  Tuesday|          3|   12|  December|      4|2025|     false|
|20251231|2025-12-31|Wednesday|          4|   12|  December|      4|2025|     false|
+--------+----------+---------+-----------+-----+----------+-------+----+----------+
only showing top 5 rows



62


In [7]:
dim_location = (
    df_lookup_raw
    .select(
        F.col(LOOKUP_COLS["location_id"]).cast("int").alias("location_id"),
        F.col(LOOKUP_COLS["borough"]).alias("borough"),
        F.col(LOOKUP_COLS["zone"]).alias("zone"),
        F.col(LOOKUP_COLS["service_zone"]).alias("service_zone"),
    )
    .dropDuplicates(["location_id"])
)

dim_location.show(5)
print(dim_location.count())

+-----------+---------+-----------------+------------+
|location_id|  borough|             zone|service_zone|
+-----------+---------+-----------------+------------+
|         12|Manhattan|     Battery Park| Yellow Zone|
|         13|Manhattan|Battery Park City| Yellow Zone|
|         14| Brooklyn|        Bay Ridge|   Boro Zone|
|         18|    Bronx|     Bedford Park|   Boro Zone|
|         38|   Queens|  Cambria Heights|   Boro Zone|
+-----------+---------+-----------------+------------+
only showing top 5 rows

265


In [8]:
attr_cols = [
    "shared_request_flag", "shared_match_flag",
    "access_a_ride_flag", "wav_request_flag", "wav_match_flag",
]

distinct_attrs = df_trips.select(*attr_cols).distinct()

w = Window.orderBy(*attr_cols)

dim_trip_attributes = (
    distinct_attrs
    .withColumn("trip_attribute_id", F.row_number().over(w).cast("int"))
    .select(
        "trip_attribute_id",
        "shared_request_flag", "shared_match_flag", "access_a_ride_flag",
        "wav_request_flag", "wav_match_flag",
    )
)

dim_trip_attributes.show(5, truncate=False)
print(dim_trip_attributes.count())

2026-09-03 09:27:45,120 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------------+-------------------+-----------------+------------------+----------------+--------------+
|trip_attribute_id|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+-------------------+-----------------+------------------+----------------+--------------+
|1                |N                  |N                |N                 |N               |N             |
|2                |N                  |N                |N                 |N               |Y             |
|3                |N                  |N                |N                 |Y               |Y             |
|4                |N                  |N                |Y                 |N               |N             |
|5                |N                  |N                |Y                 |N               |Y             |
+-----------------+-------------------+-----------------+------------------+----------------+--------------+
only showing top 5 

18


In [9]:
raw_dates_sample = df_weather_raw.select(WEATHER_COLS["date"]).limit(5).collect()
print(raw_dates_sample)

parsed_check = df_weather_raw.select(
    F.col(WEATHER_COLS["date"]).alias("raw_date"),
    F.substring(F.col(WEATHER_COLS["date"]), 1, 10).cast("date").alias("parsed_date"),
)
total_rows = parsed_check.count()
null_after_parse = parsed_check.where(F.col("parsed_date").isNull()).count()
print(total_rows, null_after_parse)

if null_after_parse > 0:
    parsed_check.where(F.col("parsed_date").isNull()).show(5, truncate=False)

[Row(Date='2025-12-01'), Row(Date='2025-12-01'), Row(Date='2025-12-01'), Row(Date='2025-12-01'), Row(Date='2025-12-01')]
48 0


In [10]:
dim_weather = (
    df_weather_raw
    .select(
        F.substring(F.col(WEATHER_COLS["date"]), 1, 10).cast("date").alias("date"),
        F.col(WEATHER_COLS["hour"]).cast("int").alias("hour"),
        F.col(WEATHER_COLS["temperature_2m"]).cast("float").alias("temperature_2m"),
        F.col(WEATHER_COLS["precipitation"]).cast("float").alias("precipitation"),
        F.col(WEATHER_COLS["snowfall"]).cast("float").alias("snowfall"),
        F.col(WEATHER_COLS["timezone"]).alias("timezone"),
    )
    .withColumn(
        "date_hour_id",
        (F.date_format("date", "yyyyMMdd").cast("bigint") * 100 + F.col("hour"))
    )
    .dropDuplicates(["date_hour_id"])
    .select("date_hour_id", "date", "hour", "temperature_2m",
            "precipitation", "snowfall", "timezone")
)

dim_weather.show(5)
print(dim_weather.count())

+------------+----------+----+--------------+-------------+--------+----------------+
|date_hour_id|      date|hour|temperature_2m|precipitation|snowfall|        timezone|
+------------+----------+----+--------------+-------------+--------+----------------+
|  2025120106|2025-12-01|   6|           2.3|          0.0|     0.0|America/New_York|
|  2025120121|2025-12-01|  21|           0.8|          0.0|     0.0|America/New_York|
|  2026010103|2026-01-01|   3|          -0.2|          0.0|     0.0|America/New_York|
|  2026010115|2026-01-01|  15|          -4.0|          0.0|     0.0|America/New_York|
|  2026010119|2026-01-01|  19|          -6.4|          0.0|     0.0|America/New_York|
+------------+----------+----+--------------+-------------+--------+----------------+
only showing top 5 rows

48


In [11]:
fact_base = (
    df_trips
    .withColumn("pickup_date_id", F.date_format(F.to_date("pickup_datetime"), "yyyyMMdd").cast("int"))
    .withColumn(
        "date_hour_id",
        (F.date_format(F.to_date("pickup_datetime"), "yyyyMMdd").cast("bigint") * 100
         + F.hour("pickup_datetime"))
    )
    .withColumn("trip_id", F.monotonically_increasing_id().cast("bigint"))
)

fact_taxi_trips = (
    fact_base
    .join(dim_trip_attributes, on=attr_cols, how="left")
    .select(
        "trip_id",
        "pickup_date_id",
        "date_hour_id",
        F.col("pickup_location_id").cast("int").alias("pickup_location_id"),
        F.col("dropoff_location_id").cast("int").alias("dropoff_location_id"),
        "trip_attribute_id",
        "hvfhs_license_num",
        "dispatching_base_num",
        "originating_base_num",
        F.col("request_datetime").cast("timestamp").alias("request_datetime"),
        F.col("on_scene_datetime").cast("timestamp").alias("on_scene_datetime"),
        F.col("pickup_datetime").cast("timestamp").alias("pickup_datetime"),
        F.col("dropoff_datetime").cast("timestamp").alias("dropoff_datetime"),
        F.col("trip_miles").cast("float").alias("trip_miles"),
        F.col("trip_time").cast("int").alias("trip_time"),
        F.col("base_passenger_fare").cast("float").alias("base_passenger_fare"),
        F.col("tolls").cast("float").alias("tolls"),
        F.col("bcf").cast("float").alias("bcf"),
        F.col("sales_tax").cast("float").alias("sales_tax"),
        F.col("congestion_surcharge").cast("float").alias("congestion_surcharge"),
        F.col("airport_fee").cast("float").alias("airport_fee"),
        F.col("tips").cast("float").alias("tips"),
        F.col("driver_pay").cast("float").alias("driver_pay"),
        F.col("cbd_congestion_fee").cast("float").alias("cbd_congestion_fee"),
    )
)

fact_taxi_trips.show(5)
print(fact_taxi_trips.count())

2026-09-03 09:27:55,458 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+------------+--------------+------------+------------------+-------------------+-----------------+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+-----+----------+------------------+
|     trip_id|pickup_date_id|date_hour_id|pickup_location_id|dropoff_location_id|trip_attribute_id|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee| tips|driver_pay|cbd_congestion_fee|
+------------+--------------+------------+------------------+-------------------+-----------------+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+----------+--------

43048810


In [12]:
def orphan_count(fact_df, fact_key, dim_df, dim_key, name):
    orphans = (
        fact_df.select(fact_key).distinct()
        .join(dim_df.select(dim_key), fact_df[fact_key] == dim_df[dim_key], "left_anti")
    )
    n = orphans.count()
    print(name, n)
    return n

orphan_count(fact_taxi_trips, "pickup_date_id", dim_date, "date_id", "fact_dim_date")
orphan_count(fact_taxi_trips, "pickup_location_id", dim_location, "location_id", "fact_dim_location_pu")
orphan_count(fact_taxi_trips, "dropoff_location_id", dim_location, "location_id", "fact_dim_location_do")
orphan_count(fact_taxi_trips, "trip_attribute_id", dim_trip_attributes, "trip_attribute_id", "fact_dim_trip_attributes")
orphan_count(fact_taxi_trips, "date_hour_id", dim_weather, "date_hour_id", "fact_dim_weather")

fact_dim_date 0


fact_dim_location_pu 0


2026-09-03 09:33:21,345 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-09-03 09:33:21,347 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


fact_dim_location_do 0


fact_dim_trip_attributes 0


fact_dim_weather 1440


1440

In [13]:
dim_date.write.mode("overwrite").format("parquet").saveAsTable("nyc_tlc_dwh.dim_date")
dim_location.write.mode("overwrite").format("parquet").saveAsTable("nyc_tlc_dwh.dim_location")
dim_trip_attributes.write.mode("overwrite").format("parquet").saveAsTable("nyc_tlc_dwh.dim_trip_attributes")
dim_weather.write.mode("overwrite").format("parquet").saveAsTable("nyc_tlc_dwh.dim_weather")

(
    fact_taxi_trips
    .repartition(8, "pickup_date_id")
    .write
    .mode("overwrite")
    .format("parquet")
    .partitionBy("pickup_date_id")
    .saveAsTable("nyc_tlc_dwh.fact_taxi_trips")
)

spark.sql("SHOW TABLES IN nyc_tlc_dwh").show(truncate=False)


2026-09-03 09:35:38,782 WARN session.SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-09-03 09:35:38,904 WARN conf.HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
2026-09-03 09:35:38,904 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-03 09:35:38,905 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist
2026-09-03 09:35:39,838 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-09-03 09:35:46,943 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+-------------------+-----------+
|database   |tableName          |isTemporary|
+-----------+-------------------+-----------+
|nyc_tlc_dwh|dim_date           |false      |
|nyc_tlc_dwh|dim_location       |false      |
|nyc_tlc_dwh|dim_trip_attributes|false      |
|nyc_tlc_dwh|dim_weather        |false      |
|nyc_tlc_dwh|fact_taxi_trips    |false      |
+-----------+-------------------+-----------+



In [14]:
spark.sql("""
    SELECT d.year, d.month_name, l.borough, COUNT(*) AS trips, ROUND(AVG(f.driver_pay), 2) AS avg_driver_pay
    FROM nyc_tlc_dwh.fact_taxi_trips f
    JOIN nyc_tlc_dwh.dim_date d ON f.pickup_date_id = d.date_id
    JOIN nyc_tlc_dwh.dim_location l ON f.pickup_location_id = l.location_id
    GROUP BY d.year, d.month_name, l.borough
    ORDER BY trips DESC
    LIMIT 10
""").show(truncate=False)

+----+----------+-------------+-------+--------------+
|year|month_name|borough      |trips  |avg_driver_pay|
+----+----------+-------------+-------+--------------+
|2025|December  |Manhattan    |7863304|25.21         |
|2026|January   |Manhattan    |7306653|21.59         |
|2025|December  |Brooklyn     |6059799|18.33         |
|2026|January   |Brooklyn     |5916553|17.71         |
|2025|December  |Queens       |4883023|22.12         |
|2026|January   |Queens       |4591439|21.85         |
|2025|December  |Bronx        |2936288|16.69         |
|2026|January   |Bronx        |2786538|16.67         |
|2025|December  |Staten Island|364957 |18.09         |
|2026|January   |Staten Island|338158 |17.46         |
+----+----------+-------------+-------+--------------+

